# Incomplete rules and several outputs

Two features that compose naturally, because both are *per rule, per output*:

- **Incomplete rules.** RIMER (Yang et al. 2006, Eq. 3) requires only that a rule's belief degrees sum to *at most* one. Whatever is left over is the rule's **ignorance** about its consequent, and the evidential reasoning combination carries it through to the prediction rather than discarding it.
- **Several consequent attributes.** One rule base can predict more than one output, each with its own grades, because objectives generally have their own scales and units.

The first changes what a prediction *is*: with ignorance in play a single number is no longer the whole answer, and the model reports an interval alongside it. The second changes how many of them you get.

For the mathematics, see the [training guide](https://desdeo-brb.readthedocs.io/en/latest/training/). This notebook is about what the two look like in use.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from desdeo_brb import BRBModel, RuleBase

## Part 1: an expert who is not sure everywhere

Consider an expert assessing the risk of a process as a function of a single control setting $x \in [0, 4]$. They know the process well at low and high settings. In the middle, where the process rarely runs, they are genuinely unsure, and the honest thing to record is that they do not know, rather than a confident-looking guess.

A classical rule base cannot express this: its belief degrees must sum to one, so the vagueness has to be spread over the grades as if it were knowledge. Here we let the middle rules assign only part of their belief.

In [ ]:
# Risk is assessed on three grades: Low (0), Medium (0.5), High (1).
prv = [np.array([0.0, 1.0, 2.0, 3.0, 4.0])]
crv = np.array([0.0, 0.5, 1.0])

# Rules 0, 1 and 4 are confident. Rules 2 and 3 assign only part of their
# belief: the rest is ignorance about what happens in the middle of the range.
belief_degrees = np.array(
    [
        [0.90, 0.10, 0.00],  # x=0: confidently Low
        [0.60, 0.40, 0.00],  # x=1: mostly Low
        [0.20, 0.30, 0.00],  # x=2: half the belief assigned, half unknown
        [0.00, 0.30, 0.20],  # x=3: half the belief assigned, half unknown
        [0.00, 0.10, 0.90],  # x=4: confidently High
    ]
)

expert = RuleBase(
    precedent_referential_values=prv,
    consequent_referential_values=crv,
    belief_degrees=belief_degrees,
    rule_weights=np.full(5, 1 / 5),
    attribute_weights=np.ones((5, 1)),
    rule_antecedent_indices=np.array([[0], [1], [2], [3], [4]]),
)

model = BRBModel(prv, crv, rule_base=expert)
print(expert.describe_all_rules(attribute_names=["x"], consequent_name="Risk"))

`RuleBase.ignorance` reads the unassigned mass straight off the rule base: zero for a rule that commits fully, one for a rule that says nothing at all.

In [ ]:
print("ignorance per rule:", expert.ignorance)
print("is the rule base complete?", expert.is_complete)

### What ignorance does to a prediction

With an incomplete assessment the prediction is no longer a single number. The unassigned belief could belong to *any* grade, so the output is only known to lie in an interval: give all of it to the least preferred grade for the lower bound, to the most preferred one for the upper bound. This is the utility interval of Yang and Xu (2002, Section II-H).

`InferenceResult.output` is the midpoint of that interval, the **average expected utility**, and `utility_bounds` is the interval itself. The width of the interval is how much the model is declining to say.

In [ ]:
X = np.linspace(0, 4, 200).reshape(-1, 1)
result = model.predict(X)
lower, upper = result.utility_bounds

plt.figure(figsize=(8, 4))
plt.fill_between(
    X[:, 0], lower, upper, alpha=0.25, color="tab:blue", label="utility interval"
)
plt.plot(X[:, 0], result.output, "b-", linewidth=2, label="prediction (midpoint)")
plt.plot(prv[0], np.zeros(5), "k|", markersize=12, label="referential values")
plt.xlabel("x")
plt.ylabel("Risk")
plt.title("The interval widens exactly where the expert declined to commit")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The band pinches shut at the ends, where the expert was confident, and opens in the middle, where they were not. That shape is the point: the model reports *where* it does not know, instead of averaging the uncertainty away into a smooth curve that looks equally trustworthy everywhere.

A single sample shows the same thing numerically.

In [ ]:
for x in (0.0, 2.5, 4.0):
    one = model.predict(np.array([[x]]))
    low, high = one.utility_bounds
    print(
        f"x={x:.1f}  risk={one.output[0]:.3f}  "
        f"in [{low[0]:.3f}, {high[0]:.3f}]  ignorance={one.ignorance[0]:.3f}"
    )

### Training without throwing the ignorance away

`fit()` decides whether the trained rules may stay incomplete. The default, `allow_incomplete=None`, follows the rule base it is given: one that arrives incomplete keeps that freedom, so training does not silently discard vagueness an expert deliberately expressed; one that arrives complete stays complete, so it gains no ignorance nobody asked for.

Pass `True` or `False` to override. Here we train the expert rule base against data, once each way, and compare what survives.

In [ ]:
rng = np.random.default_rng(0)
X_train = rng.uniform(0, 4, size=(120, 1))
y_train = np.clip(X_train[:, 0] / 4.0 + rng.normal(0, 0.05, size=120), 0.0, 1.0)


def train(allow_incomplete):
    m = BRBModel(prv, crv, rule_base=expert.model_copy(deep=True))
    m.fit(X_train, y_train, allow_incomplete=allow_incomplete, n_restarts=1)
    return m


kept = train(None)  # follows the rule base: stays free to be vague
forced = train(False)  # every rule must commit all of its belief

for label, m in [("allow_incomplete=None", kept), ("allow_incomplete=False", forced)]:
    print(f"{label:24s} MSE={-m.score(X_train, y_train):.5f}  "
          f"complete={m.rule_base.is_complete}  "
          f"ignorance={np.round(m.rule_base.ignorance, 3)}")

Both reach essentially the same error, but they say different things. The model allowed to stay incomplete keeps real ignorance on several rules: this data is equally well explained by a range of assessments, and it records that rather than picking one and presenting it as settled. Forcing completeness does not make the model better informed, it only removes its ability to say so.

Ignorance is not a free way to lower the loss, either. Total ignorance predicts the midpoint of the utility range, so declining to commit only helps a model that is already worse than predicting the mean.

## Part 2: several consequent attributes

Real decisions rarely have one objective. A rule base can carry several, each with its own grades: `crv` becomes a list of arrays, one per output.

Take a process with two inputs, temperature and pressure, and two outputs on very different scales: a **yield** in $[0, 1]$ and a **cost** in hundreds of euros.

In [ ]:
def yield_of(temp, pressure):
    return np.clip(0.3 + 0.5 * temp - 0.2 * pressure, 0.0, 1.0)


def cost_of(temp, pressure):
    return 100.0 + 250.0 * temp + 120.0 * pressure


prv2 = [np.array([0.0, 0.5, 1.0]), np.array([0.0, 0.5, 1.0])]
crv2 = [
    np.array([0.0, 0.5, 1.0]),          # Yield: three grades
    np.array([100.0, 300.0, 500.0]),    # Cost: three grades, quite different units
]

process = BRBModel(prv2, crv2, initial_rule_fn=lambda x: (yield_of(*x), cost_of(*x)))
print("rules:", process.rule_base.n_rules)
print("outputs:", process.rule_base.n_outputs)
print("grades per output:", process.rule_base.group_sizes)

The grades of all outputs are stored concatenated, and `consequent_group_sizes` says where one ends and the next begins. You rarely need to slice by hand: `consequent_values(o)` and `beliefs_for(o)` address one output at a time.

In [ ]:
for o, name in enumerate(["Yield", "Cost"]):
    print(f"{name}: grades {process.rule_base.consequent_values(o)}")
    print(f"       rule 4 believes {np.round(process.rule_base.beliefs_for(o)[4], 3)}")

Training takes a target of shape `(n_samples, n_outputs)`. By default each output's residual is divided by the span of its own grades before squaring, so an objective measured in hundreds does not crowd out one measured in tenths: pass `scale_outputs=False` for the raw sum of squared errors.

In [ ]:
X2 = rng.uniform(0, 1, size=(150, 2))
y2 = np.column_stack([yield_of(X2[:, 0], X2[:, 1]), cost_of(X2[:, 0], X2[:, 1])])

process.fit(X2, y2, n_restarts=1)

pred = process.predict_values(X2[:5])
print("predicted shape:", pred.shape)
print(np.round(pred, 2))
print("true:")
print(np.round(y2[:5], 2))

The activation weights depend only on the antecedents, so they are computed once and the evidential reasoning combination runs once per output. What a rule believes about yield places no constraint on what it believes about cost.

Both explainability helpers take one name per output, and keep the two distributions apart.

In [ ]:
print(process.rule_base.describe_rule(
    4, attribute_names=["Temp", "Pressure"], consequent_name=["Yield", "Cost"]
))

In [ ]:
print(process.explain(
    np.array([[0.7, 0.3]]),
    top_k=3,
    attribute_names=["Temp", "Pressure"],
    consequent_name=["Yield", "Cost"],
))

## Part 3: the two together

Completeness is per rule *per output*, so a rule base can be confident about one objective and vague about another. That is often the realistic case: a process engineer may know the yield curve well while having only a rough feel for cost.

In [ ]:
mixed = process.rule_base.model_copy(deep=True)

# Halve the cost beliefs of the three hottest rules: the expert is confident
# about their yield and only half-committed about their cost.
cost_block = mixed.consequent_slices[1]
bd = mixed.belief_degrees.copy()
bd[-3:, cost_block] *= 0.5
mixed.belief_degrees = bd

print("ignorance per rule, one column per output:")
print(np.round(mixed.ignorance, 3))

In [ ]:
mixed_model = BRBModel(prv2, crv2, rule_base=mixed)
res = mixed_model.predict(np.array([[0.9, 0.2]]))
low, high = res.utility_bounds

for o, name in enumerate(["Yield", "Cost"]):
    print(
        f"{name:6s} {res.output[0, o]:8.2f}  "
        f"in [{low[0, o]:.2f}, {high[0, o]:.2f}]  "
        f"ignorance={res.ignorance[0, o]:.3f}"
    )

Yield comes back as a point, its bounds coinciding with the prediction because every rule committed fully. Cost comes back as an interval. The model is precise where it can be and explicit where it cannot, in the same prediction.

## Summary

- A rule's belief degrees may sum to less than one; the shortfall is its **ignorance**, readable as `RuleBase.ignorance` and `InferenceResult.ignorance`.
- An incomplete assessment bounds the prediction rather than fixing it. `InferenceResult.utility_bounds` gives the interval and `output` its midpoint.
- `fit(allow_incomplete=...)` chooses whether training may leave rules vague; the default follows the rule base it was given.
- Several outputs are declared by passing one array of grades per output. Each keeps its own scale, `scale_outputs` keeps the wider ones from dominating the fit, and completeness is tracked per rule per output.
- `describe_rule()` and `explain()` take one consequent name per output and keep the distributions apart.

### References

- Yang, J.-B., Liu, J., Wang, J., Sii, H.-S., & Wang, H.-W. (2006). Belief rule-base inference methodology using the evidential reasoning approach — RIMER. *IEEE Transactions on Systems, Man, and Cybernetics — Part A*, 36(2), 266–285.
- Yang, J.-B., & Xu, D.-L. (2002). On the evidential reasoning algorithm for multiattribute decision analysis under uncertainty. *IEEE Transactions on Systems, Man, and Cybernetics — Part A*, 32(3), 289–304.
- Yang, J.-B., Liu, J., Xu, D.-L., Wang, J., & Wang, H.-W. (2007). Optimization models for training belief-rule-based systems. *IEEE Transactions on Systems, Man, and Cybernetics — Part A*, 37(4), 569–585.